# V2 - Phase 5 - Couche 2 : detection d'anomalie

**Objectif.** Entrainer un **Isolation Forest** sur les features V2 *sans utiliser les labels*. Produire un score d'anomalie par (siren, prediction_date) + percentile global + percentile peer-group (NAF 2 premiers chiffres). Valider en montrant que les rows a fort score d'anomalie au temps T sont surrepresentees dans les evenements de risque connus a T+12m.

**Pourquoi.** Phase 4 (couche 1) prouve que les classifieurs supervises forward captent ce qui *precede les events labellises*. Mais un action-taker veut aussi etre alerte sur les patterns *atypiques* meme quand aucun label specifique ne s'applique encore. La couche 2 isole ces ecarts sans avoir besoin de savoir ce qui est 'mauvais'. Methodologie standard en detection de fraude, surveillance reseau, QC industrielle.

**Methodologie de validation.** Le modele n'est jamais entraine sur les labels. Au test, on charge les labels (continuity_risk, legal_distress_risk, radiation_risk, financial_weakness_risk -- filing_anomaly exclu car rétrogradé en dormancy_flag) et on mesure le **lift au top-K%** : si le top-5% par score d'anomalie a 3-10x la base rate du label, la couche 2 ajoute de la valeur a la couche 1.

**Critere de validation (gate).** Lift @ top-5% >= 3 pour **chacun** des 4 labels de validation. Si seul un sous-ensemble passe, on documente et on accepte (la couche 2 sert quand meme).

**Echantillon par fit (cellule 4).** 2 000 000 rows pour iteration rapide (~5-10 min). Si lift OK, re-run sans plafond (full V2 ~30-60 min) avant le deliverable.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-v2'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE_DRIVE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

DATA_LAKE_LOCAL = '/content/pfein_phase5_lake'
ARTIFACTS_LOCAL = '/content/pfein_phase5_artifacts'
ARTIFACTS_DRIVE = f'{DRIVE_ROOT}/ml-artifacts'

# Train sample size. Set to 0 (or None) for full V2 (~30-60 min fit).
MAX_ROWS_TRAIN = 2_000_000
N_ESTIMATORS  = 100
CONTAMINATION = 'auto'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)
Path(DATA_LAKE_LOCAL).mkdir(parents=True, exist_ok=True)
Path(ARTIFACTS_LOCAL).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR     =', BACKEND_DIR)
print('DATA_LAKE_LOCAL =', DATA_LAKE_LOCAL)
print('ARTIFACTS_LOCAL =', ARTIFACTS_LOCAL)
print('MAX_ROWS_TRAIN  =', MAX_ROWS_TRAIN)

In [ ]:
import os, subprocess

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

## 2. Symlink V2 features + labels dans le data-lake local

In [ ]:
features_local = Path(DATA_LAKE_LOCAL) / 'features'
features_local.mkdir(parents=True, exist_ok=True)

shared = ['company_year_features_v2', 'risk_labels_v2']
for t in shared:
    src = Path(DATA_LAKE_DRIVE) / 'features' / t
    dst = features_local / t
    if dst.is_symlink() or dst.exists():
        if dst.is_symlink():
            dst.unlink()
        else:
            import shutil as _sh; _sh.rmtree(dst)
    if not src.exists():
        print(f'ERR Missing source on Drive: {src}')
        continue
    os.symlink(src, dst)
    print(f'symlinked {dst} -> {src}')

print('\nLocal data-lake tree:')
!ls -la "$DATA_LAKE_LOCAL/features"

## 3. Lancer le fit Isolation Forest

- Le modele apprend la structure du jeu sur train 2017-2022 (echantillon hash-deterministe MAX_ROWS_TRAIN).
- Il score 2023 (full population, pas d'echantillonnage cote test) et joint les labels de risk_labels_v2 *seulement pour valider*.
- Output: model.joblib, test_scores.parquet (siren, prediction_year, anomaly_score, global_percentile, peer_percentile, naf_prefix), metadata.json, run_summary.md.

In [ ]:
import shlex, subprocess, sys, time, json

cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.v2.train_anomaly_detector',
    '--data-lake-dir',     DATA_LAKE_LOCAL,
    '--artifacts-dir',     ARTIFACTS_LOCAL,
    '--train-start-year', '2017',
    '--train-end-year',   '2022',
    '--test-year',        '2023',
    '--n-estimators',     str(N_ESTIMATORS),
    '--contamination',    str(CONTAMINATION),
]
if MAX_ROWS_TRAIN:
    cmd += ['--max-rows', str(MAX_ROWS_TRAIN)]
else:
    cmd += ['--no-cap']

print(' '.join(shlex.quote(p) for p in cmd))
print()
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-4000:])
if result.returncode != 0:
    print('STDERR (tail):')
    print(result.stderr[-4000:])
    raise SystemExit(f'Training failed with code {result.returncode}')
print(f'\nDuree totale: {elapsed/60:.1f} min')

meta_path = Path(ARTIFACTS_LOCAL) / 'v2' / 'anomaly_detector' / 'metadata.json'
meta = json.loads(meta_path.read_text(encoding='utf-8'))

## 4. Lecture des resultats de validation

Pour chaque label de validation (continuity, legal_distress, radiation, financial_weakness) :
- **AP / AUC** de l'anomaly score contre le label (full ranking)
- **Lift au top-1%, top-5%, top-10%** : combien de fois la classe positive est sur-representee dans les top-K% les plus anormaux

Le lift est la metrique action-taker. AP / AUC sont attendus *plus bas* que la couche 1 (qui a vu les labels) -- ce qui compte est que le lift soit franchement > 1 (= mieux que random).

In [ ]:
import pandas as pd

v = meta['validation']
gate = v['gate']

rows = []
for label in [
    'continuity_risk_12m_label',
    'legal_distress_risk_12m_label',
    'radiation_risk_12m_label',
    'financial_weakness_risk_12m_label',
]:
    d = v[label]
    rows.append({
        'label'        : label.replace('_12m_label', ''),
        'base_rate'    : d['base_rate'],
        'AP_anomaly'   : d['ap_anomaly_vs_label'],
        'AUC_anomaly'  : d['auc_anomaly_vs_label'],
        'lift@1%'      : d['top_1pct']['lift'],
        'lift@5%'      : d['top_5pct']['lift'],
        'lift@10%'     : d['top_10pct']['lift'],
        'recall@5%'    : d['top_5pct']['recall'],
        'precision@5%' : d['top_5pct']['precision'],
    })
summary = pd.DataFrame(rows).set_index('label')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(summary.to_string())
print()
print(f'Gate ({gate["criterion"]}): ' + ('[OK] PASS' if gate['pass'] else '[KO] FAIL'))
for lbl, lift in gate['per_label_top_5pct_lift'].items():
    flag = '[OK]' if lift >= 3.0 else '[ko]'
    print(f'   {flag}  {lbl}: lift @ top-5% = {lift:.2f}x')

## 5. Comparaison vs baseline supervise Phase 4

Pour chaque label, on charge les test_predictions.parquet de Phase 4 (couche 1, classifieur supervise sur ce label) et on compare :
- **AP couche 1 supervisee** (entrainee sur ce label)
- **AP couche 2 anomalie** (sur le meme test, *sans avoir vu* le label)

L'anomaly doit perdre en AP pure (c'est le prix du non-supervise) MAIS son lift au top-5% peut etre comparable et son champ d'application est plus large (capte des patterns en dehors du label entraine).

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

phase4_root = Path(ARTIFACTS_DRIVE) / 'v2' / 'per_label'

compare = []
for label in [
    'continuity_risk_12m_label',
    'legal_distress_risk_12m_label',
    'radiation_risk_12m_label',
    'financial_weakness_risk_12m_label',
]:
    short = label.replace('_12m_label', '')
    p1_preds_path = phase4_root / label / 'test_predictions.parquet'
    if not p1_preds_path.exists():
        print(f'WARN: {p1_preds_path} not found, skipping')
        continue
    p1_preds = pd.read_parquet(p1_preds_path)
    p1_ap  = average_precision_score(p1_preds['y_true'], p1_preds['y_score'])
    p1_auc = roc_auc_score(p1_preds['y_true'], p1_preds['y_score'])
    p2_ap  = v[label]['ap_anomaly_vs_label']
    p2_auc = v[label]['auc_anomaly_vs_label']
    compare.append({
        'label'         : short,
        'Layer1_AP'     : p1_ap,
        'Layer1_AUC'    : p1_auc,
        'Layer2_AP'     : p2_ap,
        'Layer2_AUC'    : p2_auc,
        'AP_ratio_L2/L1': p2_ap / p1_ap if p1_ap else None,
    })

cmp_df = pd.DataFrame(compare).set_index('label')
print(cmp_df.to_string())
print()
print('Lecture : Layer1 a un AP plus haut (il a vu les labels), Layer2 plus bas. ')
print('Le ratio AP_L2/AP_L1 mesure combien le non-supervise se rapproche du supervise.')
print('Si ratio > 0.3 : la couche 2 capte une part substantielle du signal sans labels.')
print('Si ratio < 0.1 : la couche 2 est decorrelee de ce label specifique -- normal pour ')
print('                 un signal d\'anomalie generaliste, mais a documenter.')

## 6. Courbe rappel-cumule par label

Pour chaque label, on trie les rows par score d'anomalie decroissant et on trace le rappel cumule en fonction de la fraction de population flagguee. Plus la courbe monte vite, plus le score est utile.

La diagonale = un score aleatoire (pour une fraction K%, on attend K% de rappel).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

scores_path = Path(ARTIFACTS_LOCAL) / 'v2' / 'anomaly_detector' / 'test_scores.parquet'
scores_df = pd.read_parquet(scores_path)

# Re-merge with labels for plotting
import duckdb
con = duckdb.connect()
labels_df = con.execute(f"""
    SELECT siren, prediction_year,
           continuity_risk_12m_label,
           legal_distress_risk_12m_label,
           radiation_risk_12m_label,
           financial_weakness_risk_12m_label
    FROM read_parquet('{DATA_LAKE_LOCAL}/features/risk_labels_v2/**/*.parquet', union_by_name=true)
    WHERE prediction_year = 2023
""").df()
con.close()

merged = scores_df.merge(labels_df, on=['siren', 'prediction_year'], how='inner').sort_values('anomaly_score', ascending=False)
print(f'Merged rows: {len(merged):,}')

fig, ax = plt.subplots(figsize=(8, 5))
for label in [
    'continuity_risk_12m_label',
    'legal_distress_risk_12m_label',
    'radiation_risk_12m_label',
    'financial_weakness_risk_12m_label',
]:
    y = merged[label].astype(int).to_numpy()
    total_positives = int(y.sum())
    if not total_positives:
        continue
    cumulative_positives = np.cumsum(y)
    cumulative_recall = cumulative_positives / total_positives
    fraction_flagged = (np.arange(len(y)) + 1) / len(y)
    ax.plot(fraction_flagged, cumulative_recall, label=label.replace('_12m_label', ''))

ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.6, label='aleatoire')
ax.set_xlabel('Fraction de la population flaggee (du plus anormal au moins)')
ax.set_ylabel('Rappel cumule du label')
ax.set_title('Anomaly detector : rappel cumule par label (test 2023)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 7. Histogramme du score + segmentation par tier

Visualise la distribution du score d'anomalie. Le tier (amber / red) sera determine en Phase 9 selon les percentiles de la distribution train (e.g. red = top 1%, amber = top 5%).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(scores_df['anomaly_score'], bins=100, color='steelblue', alpha=0.8)
axes[0].axvline(scores_df['anomaly_score'].quantile(0.95), color='orange', linestyle='--', label='top 5% (amber)')
axes[0].axvline(scores_df['anomaly_score'].quantile(0.99), color='red', linestyle='--', label='top 1% (red)')
axes[0].set_xlabel('anomaly_score (higher = more anomalous)')
axes[0].set_ylabel('count')
axes[0].set_title('Distribution du score d\'anomalie (test 2023)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(scores_df['global_percentile'], bins=100, color='steelblue', alpha=0.8, label='global')
axes[1].hist(scores_df['peer_percentile'], bins=100, color='salmon', alpha=0.5, label='peer (NAF 2 prefix)')
axes[1].set_xlabel('percentile (100 = most anomalous)')
axes[1].set_ylabel('count')
axes[1].set_title('Percentile global vs peer-group')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nPeer groups (NAF 2-digit prefix): {scores_df["naf_prefix"].nunique()} buckets')
print(f'Top 10 NAF prefixes par taille:')
print(scores_df.groupby('naf_prefix').size().sort_values(ascending=False).head(10).to_string())

## 8. Sync ml-artifacts/v2/anomaly_detector/ vers Drive

In [ ]:
import shutil

src = Path(ARTIFACTS_LOCAL) / 'v2' / 'anomaly_detector'
dst = Path(ARTIFACTS_DRIVE) / 'v2' / 'anomaly_detector'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f'Copied {src} -> {dst}')
print('\nDrive contents:')
!ls -la "$ARTIFACTS_DRIVE/v2/anomaly_detector"

## 9. Reporter dans `docs/v2/v2_phase_log.md`

Copier la sortie ci-dessous dans la section Phase 5 du journal.

In [ ]:
from datetime import datetime

log = [
    f'**Date d\'execution :** {datetime.now().strftime("%Y-%m-%d")}',
    f'**Notebook :** `collabs/v2/v2_phase_5_anomaly_detection.ipynb`',
    f'**Script :** `app/tools/v2/train_anomaly_detector.py`',
    f'**Modele :** sklearn IsolationForest (n_estimators={N_ESTIMATORS}, contamination={CONTAMINATION!r})',
    f'**Train :** annees 2017-2022, {meta["train_rows"]:,} rows (max_rows = {meta["max_rows_train"]})',
    f'**Test  :** 2023, {meta["test_rows"]:,} rows (full)',
    '',
    '### Resultats validation (couche 2 vs labels held-out)',
    '',
    '| Label | Base rate | AP (anomaly->label) | AUC | Lift @1% | Lift @5% | Lift @10% |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for label in [
    'continuity_risk_12m_label',
    'legal_distress_risk_12m_label',
    'radiation_risk_12m_label',
    'financial_weakness_risk_12m_label',
]:
    d = v[label]
    log.append(f'| `{label}` | {d["base_rate"]:.4%} | '
               f'{(d["ap_anomaly_vs_label"] or 0):.4f} | {(d["auc_anomaly_vs_label"] or 0):.4f} | '
               f'{d["top_1pct"]["lift"]:.2f}x | {d["top_5pct"]["lift"]:.2f}x | {d["top_10pct"]["lift"]:.2f}x |')
log.append('')
log.append(f'**Gate** ({gate["criterion"]}): ' + ('PASSED' if gate['pass'] else 'FAILED'))
print('\n'.join(log))